# Notebook A — setup, graphs and baselines

Run every cell from top to bottom after restarting the kernel; the former 15 → 14 → 13 workaround is no longer needed.

In Kaggle, enable Internet, attach the `cosmicnet-data` dataset containing `tng100_clustered.csv` and the actual `best_model_augmented.pt`, and enable the `GITHUB_PAT` secret for private repository access. GPU is optional. Use the settings cell to select another dataset path.

Git LFS downloads are skipped because the checkpoint comes from the attached dataset. Native PyG extensions are optional: graph construction uses a radius/kNN fallback when unavailable, with the backend recorded in provenance. See the [PyG installation documentation](https://pytorch-geometric.readthedocs.io/en/2.6.1/install/installation.html).

Set `SMOKE_TEST = True` for a short integration run; those results are saved separately under `outputs/rls/smoke`. They are not research results. Normal runs retain the full split and 20 Gumbel training epochs. The attention control is a seeded, untrained MLP proxy.

Old stored outputs have been cleared. This notebook produces local artifacts only; it never commits or pushes code.

In [ ]:
# Settings — edit these before Run All.
import os
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/cosmic-net")
REPO_URL = "https://github.com/Neal-Salian/cosmic-net-f.git"
REPO_BRANCH = "fix/rl-pruning-symmetry"
SYNC_REPO = True                 # False for an already prepared local checkout.
INSTALL_MISSING = True
INPUT_DIR = None                 # None: discover one matching Kaggle dataset.
INPUT_ROOT = Path("/kaggle/input")
DEVICE = "auto"                  # "auto", "cpu", or "cuda"
SMOKE_TEST = False
SMOKE_GRAPHS_PER_SPLIT = 3
GUMBEL_EPOCHS = 20
FRACTIONS = [0.1, 0.25, 0.4, 0.6, 0.8, 1.0]
GRAPH_BACKEND = "auto"           # "auto" or "torch" (no native extensions)
OUTPUT_DIR = None                # None: REPO_DIR / outputs / rls

In [ ]:
# Dependencies — retain Kaggle's PyTorch and install only missing core packages.
import importlib
import importlib.util
import subprocess
import torch

print("Python:", sys.version.split()[0], "| torch:", torch.__version__,
      "| CUDA build:", torch.version.cuda, "| GPU available:", torch.cuda.is_available())
packages = {"torch_geometric": "torch-geometric>=2.6,<3", "yaml": "pyyaml",
            "numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
            "matplotlib": "matplotlib", "h5py": "h5py", "requests": "requests"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    if not INSTALL_MISSING:
        raise RuntimeError("Missing packages: " + ", ".join(missing))
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
    importlib.invalidate_caches()
# NNConv works without torch-scatter/torch-sparse. Do not install CPU extension
# wheels into a CUDA runtime or compile unsupported wheels during notebook setup.
import torch_geometric
print("PyG:", torch_geometric.__version__)

In [ ]:
# Repository checkout — skip LFS and never put the PAT in command-line URLs.
import base64
import tempfile

def prepare_repository(repo_dir, repo_url, branch, sync=True):
    repo_dir = Path(repo_dir).resolve()
    git_env = dict(os.environ, GIT_LFS_SKIP_SMUDGE="1", GIT_TERMINAL_PROMPT="0")
    token = ""
    encoded = ""
    if sync:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("GITHUB_PAT") or ""
        except Exception:
            # Public repositories can be cloned without a secret.
            print("GITHUB_PAT unavailable; trying unauthenticated repository access.")
        if token:
            encoded = base64.b64encode(("x-access-token:" + token).encode()).decode()
            # Git reads this transient header from the child environment only.
            count = int(git_env.get("GIT_CONFIG_COUNT", "0"))
            git_env.update({"GIT_CONFIG_COUNT": str(count + 1),
                            f"GIT_CONFIG_KEY_{count}": "http.https://github.com/.extraheader",
                            f"GIT_CONFIG_VALUE_{count}": "Authorization: Basic " + encoded})

    def git(*args):
        result = subprocess.run(["git", *map(str, args)], env=git_env,
                                capture_output=True, text=True)
        if result.returncode:
            detail = result.stderr or result.stdout
            for secret in (token, encoded):
                if secret:
                    detail = detail.replace(secret, "[redacted]")
            raise RuntimeError("Git operation failed:\n" + detail.strip()) from None
        return result.stdout.strip()

    if sync:
        if not repo_dir.exists():
            repo_dir.parent.mkdir(parents=True, exist_ok=True)
            staging = Path(tempfile.mkdtemp(prefix="cosmic-net-clone-", dir=repo_dir.parent)) / "repo"
            git("clone", "--no-checkout", "--branch", branch, repo_url, staging)
            git("-C", staging, "checkout", branch)
            staging.rename(repo_dir)  # A failed clone never becomes REPO_DIR.
        if not (repo_dir / ".git").exists():
            raise RuntimeError(f"{repo_dir} is not a Git checkout. Select a different REPO_DIR.")
        git("-C", repo_dir, "remote", "set-url", "origin", repo_url)
        git("-C", repo_dir, "fetch", "origin", branch)
        head = git("-C", repo_dir, "rev-parse", "HEAD")
        fetched = git("-C", repo_dir, "rev-parse", "FETCH_HEAD")
        current = git("-C", repo_dir, "branch", "--show-current")
        if head != fetched or current != branch:
            if git("-C", repo_dir, "status", "--porcelain", "--untracked-files=no"):
                raise RuntimeError("Cached checkout has local edits or an incomplete checkout. "
                                   "Preserve it and select a new REPO_DIR before rerunning setup.")
            git("-C", repo_dir, "checkout", branch)
            git("-C", repo_dir, "merge", "--ff-only", "FETCH_HEAD")
    if not (repo_dir / "config/config.yaml").is_file():
        raise FileNotFoundError(f"Repository files missing in {repo_dir}; run checkout first.")
    head = git("-C", repo_dir, "rev-parse", "HEAD")
    if globals().get("_A_IMPORTED_HEAD") not in (None, head):
        raise RuntimeError("Repository revision changed after imports. Restart the kernel and Run All.")
    return repo_dir, head

REPO_DIR, REPO_HEAD = prepare_repository(REPO_DIR, REPO_URL, REPO_BRANCH, SYNC_REPO)
repo_path = str(REPO_DIR)
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
os.chdir(REPO_DIR)
print("Repository:", REPO_DIR, "| HEAD:", REPO_HEAD)

In [ ]:
# Locate and validate the attached dataset before any model or graph work.
import numpy as np
import pandas as pd

def locate_inputs(input_dir, input_root):
    if input_dir is not None:
        candidates = [Path(input_dir)]
    else:
        candidates = sorted({p.parent for p in Path(input_root).rglob("tng100_clustered.csv")
                             if (p.parent / "best_model_augmented.pt").is_file()})
    if len(candidates) != 1:
        raise FileNotFoundError("Set INPUT_DIR to the dataset folder containing "
                                "tng100_clustered.csv and best_model_augmented.pt. "
                                f"Matching folders: {candidates}")
    folder = candidates[0].resolve()
    csv_path = folder / "tng100_clustered.csv"
    checkpoint_path = folder / "best_model_augmented.pt"
    for path in (csv_path, checkpoint_path):
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f"Missing or empty input: {path}")
        with path.open("rb") as stream:
            if stream.read(128).startswith(b"version https://git-lfs.github.com/spec/v1"):
                raise ValueError(f"{path} is a Git LFS pointer, not the actual data file.")
    return csv_path, checkpoint_path

CSV_PATH, CHECKPOINT_PATH = locate_inputs(INPUT_DIR, INPUT_ROOT)
catalog = pd.read_csv(CSV_PATH)
velocity_column = "vel_dispersion" if "vel_dispersion" in catalog else "velocity_dispersion"
required = ["subhalo_id", "group_id", "stellar_mass", velocity_column,
            "half_mass_radius", "metallicity", "pos_x", "pos_y", "pos_z",
            "vel_x", "vel_y", "vel_z"]
target_column = "halo_mass" if "halo_mass" in catalog else "halo_mass_log"
required.append(target_column)
missing_columns = sorted(set(required) - set(catalog.columns))
if missing_columns:
    raise ValueError(f"CSV missing required columns: {missing_columns}")
numeric = catalog[required].apply(pd.to_numeric, errors="coerce")
if catalog.empty or not np.isfinite(numeric.to_numpy()).all():
    raise ValueError("CSV must contain finite numeric features and targets in every row.")
if (numeric[[velocity_column, "half_mass_radius", "metallicity"]] < 0).any().any():
    raise ValueError("Velocity dispersion, radius and metallicity cannot be negative.")
print("CSV:", CSV_PATH, "| rows:", len(catalog), "| halos:", catalog.group_id.nunique())
print("Checkpoint:", CHECKPOINT_PATH, "| bytes:", CHECKPOINT_PATH.stat().st_size)

In [ ]:
# Configuration and imports — retain the repository's RL settings.
import copy
import yaml
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Batch, Data
from torch_geometric.loader import DataLoader
from data.loaders.base_loader import get_loader
from graph.graph_builder import GraphBuilder
from model.model import build_model
from model.physics_loss import MetricsComputer
from rls.baselines import (random_mask, degree_mask, distance_mask,
                           mass_ratio_mask, gradient_saliency_mask,
                           attention_topk_mask, GumbelEdgeMask)
from rls.sparsify import hard_mask, repair_connectivity
from rls.provenance import record_backbone, require_backbone_label

_A_IMPORTED_HEAD = REPO_HEAD
cfg = yaml.safe_load((REPO_DIR / "config/config.yaml").read_text())
cfg["data"].update(source="tng", num_workers=0, batch_size=16)
cfg["data"]["tng"]["clustered_file"] = str(CSV_PATH)
if DEVICE not in ("auto", "cpu", "cuda"):
    raise ValueError("DEVICE must be auto, cpu or cuda.")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA requested but unavailable. Enable a Kaggle GPU or use DEVICE='cpu'.")
device = torch.device("cuda" if DEVICE == "cuda" or
                      (DEVICE == "auto" and torch.cuda.is_available()) else "cpu")
if GRAPH_BACKEND not in ("auto", "torch"):
    raise ValueError("GRAPH_BACKEND must be auto or torch.")
if not FRACTIONS or any(not 0 < f <= 1 for f in FRACTIONS):
    raise ValueError("FRACTIONS must contain values in (0, 1].")
if GUMBEL_EPOCHS < 1 or SMOKE_GRAPHS_PER_SPLIT < 2:
    raise ValueError("Use at least one Gumbel epoch and two smoke graphs per split.")
OUT = Path(OUTPUT_DIR) if OUTPUT_DIR is not None else REPO_DIR / "outputs/rls"
if SMOKE_TEST:
    OUT = OUT / "smoke"
OUT.mkdir(parents=True, exist_ok=True)
cfg["data"]["tng"]["cache_dir"] = str(OUT / "tng_cache")
# Mark this run as incomplete until the final cell records all artifacts.
import json
provenance_path = OUT / "provenance.json"
previous = json.loads(provenance_path.read_text()) if provenance_path.exists() else {}
previous["A"] = {"stage": "A", "status": "running", "repo": {"head": REPO_HEAD}}
provenance_path.write_text(json.dumps(previous, indent=2))
seed = int(cfg.get("seed", 42))
torch.manual_seed(seed)
np.random.seed(seed)
print("Device:", device, "| run mode:", "SMOKE" if SMOKE_TEST else "FULL", "| outputs:", OUT)

In [ ]:
# Load only the supplied checkpoint; use its own architecture and graph settings.
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
checkpoint_config = checkpoint.get("config") if isinstance(checkpoint, dict) else None
if checkpoint_config and "model" in checkpoint_config:
    cfg["model"] = copy.deepcopy(checkpoint_config["model"])
    if "graph" in checkpoint_config:
        cfg["graph"] = copy.deepcopy(checkpoint_config["graph"])
cfg["model"]["mc_samples"] = 30
if cfg["graph"].get("hierarchical", False):
    raise ValueError("Notebook A supports single-level graphs; hierarchical mode is only a scaffold.")
state_dict = checkpoint.get("model_state_dict", checkpoint)
gnn = build_model(cfg)
try:
    gnn.load_state_dict(state_dict, strict=True)
except RuntimeError as exc:
    raise RuntimeError("Checkpoint architecture does not match CosmicNetGNN. "
                       "Supply the matching augmented checkpoint/config.\n" + str(exc)) from None
gnn = gnn.to(device).eval()
gnn.requires_grad_(False)  # Saliency still differentiates input features.
model_device = next(gnn.parameters()).device
backbone_record = record_backbone("frozen", gnn)
del checkpoint, state_dict
print("Loaded:", CHECKPOINT_PATH.name, "| dimensions:", gnn.hidden_dim, gnn.output_dim,
      "| parameters:", sum(p.numel() for p in gnn.parameters()))

In [ ]:
# Build each halo independently; fail on invalid graphs instead of dropping them.
def validate_graph(graph):
    if graph.num_nodes < 1 or graph.edge_index.shape[1] == 0:
        raise ValueError(f"Empty graph or no edges: {graph.cluster_id}")
    graph.validate(raise_on_error=True)
    if graph.x.shape[1] != gnn.node_input_dim or graph.edge_attr.shape != (
            graph.edge_index.shape[1], gnn.edge_input_dim):
        raise ValueError(f"Feature dimensions incompatible with checkpoint: {graph.cluster_id}")
    for name in ("x", "pos", "edge_attr", "y"):
        value = getattr(graph, name, None)
        if value is None or not torch.isfinite(value).all():
            raise ValueError(f"Missing or non-finite {name}: {graph.cluster_id}")
    if graph.pos.shape != (graph.num_nodes, 3):
        raise ValueError(f"Expected physical 3D positions: {graph.cluster_id}")

loader = get_loader(cfg)
halos = loader.load()
split_halos = dict(zip(("train", "val", "test"), loader.split_data(halos)))
full_split_sizes = {name: len(items) for name, items in split_halos.items()}
if any(size < 2 for size in full_split_sizes.values()):
    raise ValueError(f"Each split needs at least two halos; got {full_split_sizes}.")
if SMOKE_TEST:
    split_halos = {name: items[:SMOKE_GRAPHS_PER_SPLIT] for name, items in split_halos.items()}
torch.manual_seed(seed)
np.random.seed(seed)
builder = GraphBuilder(cfg)
split_graphs = {}
for name, items in split_halos.items():
    graphs = []
    for halo in items:
        graph = builder.build_graph(halo)
        validate_graph(graph)
        graphs.append(graph)
    split_graphs[name] = graphs
train_loader = DataLoader(split_graphs["train"], batch_size=cfg["data"]["batch_size"], shuffle=True)
val_loader = DataLoader(split_graphs["val"], batch_size=cfg["data"]["batch_size"], shuffle=False)
test_loader = DataLoader(split_graphs["test"], batch_size=cfg["data"]["batch_size"], shuffle=False)
print("Full split:", full_split_sizes, "| evaluated split:",
      {name: len(graphs) for name, graphs in split_graphs.items()},
      "| graph backend: portable torch | tie policy: inclusive")

In [ ]:
# Prediction and scoring helpers are defined before device checks and evaluation.
def checked_mask(graph, mask):
    if not isinstance(mask, torch.Tensor) or mask.dtype != torch.bool:
        raise TypeError("Baseline masks must be boolean tensors.")
    if mask.shape != (graph.edge_index.shape[1],):
        raise ValueError("Baseline mask length must match the number of edges.")
    return mask.to(graph.edge_index.device)

def pred_for_mask(graph, mask, model=None):
    model = gnn if model is None else model
    mask = checked_mask(graph, mask)
    batch = Batch.from_data_list([Data(x=graph.x,
        edge_index=graph.edge_index[:, mask], edge_attr=graph.edge_attr[mask])])
    batch = batch.to(next(model.parameters()).device)
    model.eval()
    with torch.no_grad():
        prediction, _ = model(batch)
    if prediction.numel() != 1 or not torch.isfinite(prediction).all():
        raise RuntimeError(f"Non-finite or non-scalar prediction for {graph.cluster_id}.")
    return prediction.item()

class _AttentionScorer(nn.Module):
    def __init__(self, edge_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(edge_dim, 64), nn.LeakyReLU(),
                                 nn.Linear(64, 32), nn.LeakyReLU(), nn.Linear(32, 1))

    def forward(self, edge_attr):
        with torch.no_grad():
            return self.net(edge_attr).reshape(-1)

# Fork the RNG so constructing this proxy does not reseed subsequent training.
with torch.random.fork_rng(devices=[]):
    torch.manual_seed(seed)
    attn_scorer = _AttentionScorer(gnn.edge_input_dim).to(model_device).eval()

def metrics_for(predictions, targets):
    predictions = torch.tensor(predictions, dtype=torch.float32)
    targets = torch.tensor(targets, dtype=torch.float32)
    if predictions.shape != targets.shape or predictions.numel() < 2:
        raise ValueError("Metrics need matching predictions/targets for at least two halos.")
    if not torch.isfinite(predictions).all() or not torch.isfinite(targets).all():
        raise ValueError("Metrics received non-finite values.")
    return MetricsComputer.compute_all(predictions, targets)

def save_baselines(frame):
    destination = OUT / "baselines.csv"
    temporary = destination.with_suffix(".csv.tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(destination)

In [ ]:
# A real single halo, not a batch relabelled as one graph.
g = split_graphs["test"][0].clone().to(model_device)
print("=== DEVICE CHECK ===")
print("GNN:", model_device)
print("Test graph:", {name: getattr(g, name).device for name in ("x", "edge_index", "edge_attr")})
print("Attention scorer:", next(attn_scorer.parameters()).device)
full_mask = torch.ones(g.num_edges, dtype=torch.bool, device=model_device)
print("Full-graph smoke prediction:", pred_for_mask(g, full_mask))

In [ ]:
# Attention smoke test — pass the callable, not an already computed score tensor.
mask = attention_topk_mask(g.edge_index, g.edge_attr, g.pos, 0.5, model=attn_scorer)
mask = checked_mask(g, mask)
pred = pred_for_mask(g, mask)
print("Mask device:", mask.device, "| edges:", g.num_edges,
      "| kept:", mask.sum().item(), "| prediction:", pred)

In [ ]:
# Evaluate every method at every fraction; publish only the completed table.
methods = [("random", random_mask), ("degree", degree_mask),
           ("distance", distance_mask), ("mass_ratio", mass_ratio_mask),
           ("grad_saliency", gradient_saliency_mask), ("attention_topk", attention_topk_mask)]
results = []
gnn.eval()
for frac in FRACTIONS:
    for name, mask_fn in methods:
        predictions, targets, keeps = [], [], []
        for original in split_graphs["test"]:
            graph = original.clone().to(model_device)
            kwargs = {}
            if name == "grad_saliency":
                kwargs = {"model": gnn, "x": graph.x}
            elif name == "attention_topk":
                kwargs = {"model": attn_scorer}
            elif name == "random":
                kwargs = {"seed": seed}
            # Saliency needs autograd, even though the backbone weights are frozen.
            with torch.enable_grad():
                edge_mask = mask_fn(graph.edge_index, graph.edge_attr, graph.pos, frac, **kwargs)
            edge_mask = checked_mask(graph, edge_mask)
            predictions.append(pred_for_mask(graph, edge_mask))
            targets.append(graph.y.item())
            keeps.append(edge_mask.float().mean().item())
        metrics = metrics_for(predictions, targets)
        results.append({"method": name, "frac": frac, "rmse": metrics["rmse"],
                        "r2": metrics["r2"], "keep_frac": float(np.mean(keeps))})
        print(f"{name} frac={frac:.2f}: RMSE={metrics['rmse']:.4f}", flush=True)

full_predictions, full_targets = [], []
for original in split_graphs["test"]:
    graph = original.clone().to(model_device)
    edge_mask = torch.ones(graph.num_edges, dtype=torch.bool, device=model_device)
    full_predictions.append(pred_for_mask(graph, edge_mask))
    full_targets.append(graph.y.item())
full_m = metrics_for(full_predictions, full_targets)
results.append({"method": "full", "frac": 1.0, "rmse": full_m["rmse"],
                "r2": full_m["r2"], "keep_frac": 1.0})
require_backbone_label(results, backbone_record)
baseline_df = pd.DataFrame(results)
baseline_df["run_mode"] = "smoke" if SMOKE_TEST else "full"
save_baselines(baseline_df)
print(baseline_df.to_string(index=False))

In [ ]:
# Plot completed baselines.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4.5))
for name, group in baseline_df[baseline_df.method != "full"].groupby("method"):
    ax.plot(group.keep_frac, group.rmse, "o-", label=name)
ax.axhline(full_m["rmse"], color="k", linestyle="--", label="full graph")
ax.set(xlabel="mean keep fraction", ylabel="RMSE (dex)")
ax.legend()
ax.grid(alpha=0.3)
fig.savefig(OUT / "baselines_pareto.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("Saved:", OUT / "baselines_pareto.png")

In [ ]:
# Gumbel control: train a separate backbone copy and edge-mask network jointly.
# Training scales edge features with soft masks; evaluation physically drops
# edges. These are different operations, so report this control's limits.
torch.manual_seed(seed)
gumbel_gnn = copy.deepcopy(gnn).to(model_device).requires_grad_(True).train()
gumbel_mask_net = GumbelEdgeMask(edge_dim=gnn.edge_input_dim, hidden_dim=64).to(model_device)
gumbel_mask_net.train()
gumbel_parameters = list(gumbel_gnn.parameters()) + list(gumbel_mask_net.parameters())
g_opt = torch.optim.Adam(gumbel_parameters, lr=1e-4)
gumbel_target_sparsity = float(cfg["rls"].get("target_sparsity_end", 0.4))
gumbel_epochs = min(GUMBEL_EPOCHS, 2) if SMOKE_TEST else GUMBEL_EPOCHS
gumbel_log = []
for epoch in range(gumbel_epochs):
    tau = 1.0 + (0.3 - 1.0) * epoch / max(1, gumbel_epochs - 1)
    losses, keeps = [], []
    for batch in train_loader:
        for original in batch.to_data_list():
            graph = original.to(model_device)
            p_keep = gumbel_mask_net(graph.edge_attr, hard=False, tau=tau)
            if p_keep.shape != (graph.num_edges,) or not torch.isfinite(p_keep).all():
                raise RuntimeError("Gumbel mask probabilities must be finite and one per edge.")
            data = Batch.from_data_list([Data(x=graph.x, edge_index=graph.edge_index,
                                              edge_attr=graph.edge_attr * p_keep[:, None])])
            prediction, _ = gumbel_gnn(data.to(model_device))
            loss = F.mse_loss(prediction.reshape(-1), graph.y.reshape(-1)) + (
                p_keep.mean() - gumbel_target_sparsity).square()
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite Gumbel training loss at epoch {epoch}.")
            g_opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gumbel_parameters, 1.0, error_if_nonfinite=True)
            g_opt.step()
            losses.append(loss.item())
            keeps.append(p_keep.mean().item())
    gumbel_log.append([epoch, tau, float(np.mean(losses)), float(np.mean(keeps))])
    print(f"Gumbel epoch {epoch + 1}/{gumbel_epochs}: loss={np.mean(losses):.4f}", flush=True)
torch.save(gumbel_mask_net.state_dict(), OUT / "gumbel_mask_net.pt")
torch.save(gumbel_gnn.state_dict(), OUT / "gumbel_gnn.pt")
pd.DataFrame(gumbel_log, columns=["epoch", "tau", "loss", "mean_keep_p"]).to_csv(
    OUT / "gumbel_training_log.csv", index=False)
print("Gumbel training complete.")

In [ ]:
# Gumbel evaluation — reproducible draws and replace an existing row on rerun.
gumbel_gnn.eval()
gumbel_mask_net.eval()
predictions, targets, keeps = [], [], []
cuda_devices = [model_device.index or 0] if model_device.type == "cuda" else []
with torch.random.fork_rng(devices=cuda_devices), torch.no_grad():
    torch.manual_seed(seed)
    for original in split_graphs["test"]:
        graph = original.clone().to(model_device)
        edge_mask = gumbel_mask_net(graph.edge_attr, hard=True)[:, 1] > 0.5
        if not edge_mask.any():
            edge_mask = hard_mask(gumbel_mask_net(graph.edge_attr, hard=False),
                                  cfg["rls"].get("min_keep_frac", 0.1))
        edge_mask = repair_connectivity(graph.edge_index, edge_mask)
        edge_mask = checked_mask(graph, edge_mask)
        predictions.append(pred_for_mask(graph, edge_mask, model=gumbel_gnn))
        targets.append(graph.y.item())
        keeps.append(edge_mask.float().mean().item())
gumbel_m = metrics_for(predictions, targets)
gumbel_record = record_backbone("gumbel_joint", gumbel_gnn)
gumbel_row = {"method": "gumbel", "frac": float(np.mean(keeps)),
              "rmse": gumbel_m["rmse"], "r2": gumbel_m["r2"],
              "keep_frac": float(np.mean(keeps)),
              "run_mode": "smoke" if SMOKE_TEST else "full"}
require_backbone_label([gumbel_row], gumbel_record)
baseline_df = pd.concat([baseline_df[baseline_df.method != "gumbel"],
                        pd.DataFrame([gumbel_row])], ignore_index=True)
save_baselines(baseline_df)
print(gumbel_row)

In [ ]:
# Record the actual inputs, split, environment and completion of this run.
import hashlib
import json
import platform
from datetime import datetime, timezone

def file_info(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return {"path": str(path), "size_bytes": Path(path).stat().st_size,
            "sha256": digest.hexdigest()}

provenance_path = OUT / "provenance.json"
provenance = json.loads(provenance_path.read_text()) if provenance_path.exists() else {}
provenance["A"] = {
    "stage": "A", "status": "complete", "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "repo": {"head": REPO_HEAD, "branch": REPO_BRANCH},
    "dataset": {"tng_csv": file_info(CSV_PATH), "checkpoint": file_info(CHECKPOINT_PATH)},
    "seed": seed, "python": platform.python_version(), "torch": torch.__version__,
    "pyg": torch_geometric.__version__, "device": str(model_device),
    "cuda": {"available": torch.cuda.is_available(), "version": torch.version.cuda},
    "run_mode": "smoke" if SMOKE_TEST else "full",
    "full_split_sizes": full_split_sizes, "graph_backend": "portable_torch",
    "split_cluster_ids": {name: [g.cluster_id for g in graphs] for name, graphs in split_graphs.items()},
    "config_used": cfg, "backbone": backbone_record, "gumbel_backbone": gumbel_record,
    "fractions": FRACTIONS, "gumbel_epochs": gumbel_epochs,
    "attention_control": "seeded untrained MLP proxy",
    "artifacts": {name: file_info(OUT / name) for name in
                  ("baselines.csv", "baselines_pareto.png", "gumbel_mask_net.pt",
                   "gumbel_gnn.pt", "gumbel_training_log.csv")},
}
temporary = provenance_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(provenance, indent=2))
temporary.replace(provenance_path)
print("Notebook A completed. Artifacts:", OUT)